# TOXRIC + KPGT — End-to-End Pipeline Notebook

Walks through every stage of the pipeline by calling functions from this project's `src/` and `scripts/`. 
Each section is self-describing and can be run in order from top to bottom.

**Sections:**
0. One-time setup — download TOXRIC, clone KPGT, check the pretrained checkpoint
1. Single-target TOXRIC data — what one of the 30 source CSVs looks like
2. Convert single-target → multi-target (the merge step)
3. KPGT preprocessing (graphs + fingerprints + descriptors)
4. Featurize one SMILES (inspect what the model actually sees)
5. Build the fine-tuning model (pretrained KPGT + multi-task head)
6. Training smoke-test (one batch, one optimizer step)
7. Run prediction on a few SMILES
8. Generate plain-English explanation via Claude

**Dependency tiers** (each section's first cell prints what's missing):
- 0, 1, 2: `pandas`, `numpy`, `requests`, `tqdm`
- 3, 4, 5, 6, 7: `rdkit`, `torch`, `dgl`, `dgllife`, `scipy`, `scikit-learn`
- 8: `anthropic`, `azure-identity` (+ a populated `.env`)


## Setup


In [1]:
import os, sys
from pathlib import Path

# Ensure we run from the toxicity-kpgt directory so toxpkg/ imports + relative paths work.
ROOT = Path.cwd()
if ROOT.name != 'toxicity-kpgt':
    candidate = ROOT / 'labfiles' / 'toxicity-kpgt'
    if candidate.exists():
        os.chdir(candidate)
        ROOT = Path.cwd()
print('cwd:', ROOT)

# Make src/ importable
sys.path.insert(0, str(ROOT))
print('python:', sys.version.split()[0])


cwd: /Users/abose1/Avishek_Personal_Projects/mslearn-ai-studio/labfiles/toxicity-kpgt
python: 3.12.13


## Step 0 — One-time setup

Three things only need to happen once per machine:

1. **Download TOXRIC** from Figshare (~7.7 MB, the official 30-datasets archive — DOI 27195339)
2. **Clone the KPGT repo** (`lihan97/KPGT`) so its featurizer, dataset, and model code are importable from `external/KPGT/`
3. **Check the pretrained checkpoint** — `base.pth` lives behind an anonymous Figshare share link with no public API, so this one is a manual download. The check just tells you whether the file is in place.

All three are idempotent — they skip automatically if already done.


In [2]:
from toxpkg.data_utils import download_toxric, clone_kpgt, check_kpgt_pretrained, list_toxric_subdatasets

# 1. Download TOXRIC (skipped if already extracted)
extract_dir = download_toxric(which='toxric_30_datasets.zip')
csvs = list_toxric_subdatasets(extract_dir)
print(f'\n[ok] {len(csvs)} TOXRIC endpoint CSVs at {extract_dir}')


[skip] toxric_30_datasets.zip already present at data/toxric/toxric_30_datasets.zip
[skip] data/toxric/toxric_30_datasets already extracted

[ok] 30 TOXRIC endpoint CSVs at data/toxric/toxric_30_datasets


In [3]:
# 2. Clone KPGT (skipped if .git already exists)
kpgt_dir = clone_kpgt()
print(f'[ok] KPGT at {kpgt_dir}')


[skip] KPGT already cloned at external/KPGT
[ok] KPGT at external/KPGT


In [4]:
# 3. Check pretrained base.pth — manual download required if missing
ok = check_kpgt_pretrained(kpgt_dir)
if not ok:
    print('\nFollow the instructions above, then re-run this cell. Steps 5+ need the weights.')


[ok] base.pth found at external/KPGT/models/pretrained/base/base.pth (447.1 MB)


## Step 1 — Single-target TOXRIC data

TOXRIC's `toxric_30_datasets.zip` extracts to 30 CSVs, each covering ONE toxicity endpoint. 
Schema is the same across all 30: `TAID, Name, IUPAC Name, PubChem CID, Canonical SMILES, InChIKey, Toxicity Value`. 
The `Toxicity Value` column is binary (0 = inactive, 1 = active for that endpoint).


In [5]:
import pandas as pd
from toxpkg.data_utils import inspect_csv, print_inspection

# csvs and extract_dir were populated in Step 0; reuse them throughout this section.
TOXRIC_DIR = extract_dir


In [6]:
# Inspect one endpoint to see the source schema
info = inspect_csv(csvs[csvs.index([p for p in csvs if 'Hepatotoxicity' in p.name][0])])
print_inspection(info)



=== data/toxric/toxric_30_datasets/toxric_30_datasets/Hepatotoxicity_Hepatotoxicity.csv ===
rows: 2889
columns (7): ['TAID', 'Name', 'IUPAC Name', 'PubChem CID', 'Canonical SMILES', 'InChIKey', 'Toxicity Value']
SMILES column guess: 'Canonical SMILES'
first 3 rows:
  {'TAID': 'TOX-9', 'Name': 'ethinamate', 'IUPAC Name': '(1-ethynylcyclohexyl) carbamate', 'PubChem CID': 3284.0, 'Canonical SMILES': 'C#CC1(OC(N)=O)CCCCC1', 'InChIKey': 'GXRZIMHKGDIBEW-UHFFFAOYSA-N', 'Toxicity Value': 0}
  {'TAID': 'TOX-21', 'Name': 'D-Sorbitol', 'IUPAC Name': '(2R,3R,4R,5S)-hexane-1,2,3,4,5,6-hexol', 'PubChem CID': 5780.0, 'Canonical SMILES': 'OC[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO', 'InChIKey': 'FBPFZTCFMRRESA-JGWLITMVSA-N', 'Toxicity Value': 0}
  {'TAID': 'TOX-23', 'Name': 'Perflubron', 'IUPAC Name': '1-bromo-1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-heptadecafluorooctane', 'PubChem CID': 9873.0, 'Canonical SMILES': 'FC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)Br', 'InChIKey': 'WTWWXOGTJWMJHI-UHFFFA

In [7]:
# Class distribution for this single endpoint
df_one = pd.read_csv([p for p in csvs if 'Hepatotoxicity' in p.name][0])
print('Rows:', len(df_one))
print('Label distribution:')
print(df_one['Toxicity Value'].value_counts())
df_one.head()


Rows: 2889
Label distribution:
Toxicity Value
1    1465
0    1424
Name: count, dtype: int64


,TAID,Name,IUPAC Name,PubChem CID,Canonical SMILES,InChIKey,Toxicity Value
0,TOX-9,ethinamate,(1-ethynylcyclohexyl) carbamate,3284.0,C#CC1(OC(N)=O)CCCCC1,GXRZIMHKGDIBEW-UHFFFAOYSA-N,0
1,TOX-21,D-Sorbitol,"(2R,3R,4R,5S)-hexane-1,2,3,4,5,6-hexol",5780.0,OC[C@H](O)[C@@H](O)[C@H](O)[C@H](O)CO,FBPFZTCFMRRESA-JGWLITMVSA-N,0
2,TOX-23,Perflubron,"1-bromo-1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-hept...",9873.0,FC(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F)(F)C(F...,WTWWXOGTJWMJHI-UHFFFAOYSA-N,1
3,TOX-27,NaN,NaN,NaN,NaN,NaN,0
4,TOX-28,nitroglycerin,"1,3-dinitrooxypropan-2-yl nitrate",4510.0,O=[N+]([O-])OCC(CO[N+](=O)[O-])O[N+](=O)[O-],SNIOPGDIGTZGOP-UHFFFAOYSA-N,1


In [8]:
# Summary across all 30 single-target sub-datasets
print("\n--- Summary across all sub-datasets ---")
for csv in csvs:
    quick = inspect_csv(csv, n_rows=0)
    print(f"  {csv.name:<55} rows={quick['n_rows_total']:>6}  cols={len(quick['columns'])}")



--- Summary across all sub-datasets ---
  CYP450_CYP1A2.csv                                       rows=  9690  cols=7
  CYP450_CYP2C19.csv                                      rows= 10530  cols=7
  CYP450_CYP2C9.csv                                       rows=  9841  cols=7
  CYP450_CYP2D6.csv                                       rows= 10618  cols=7
  CYP450_CYP3A4.csv                                       rows= 11288  cols=7
  Carcinogenicity_Carcinogenicity.csv                     rows=  1021  cols=7
  Cardiotoxicity_Cardiotoxicity-1.csv                     rows=  1547  cols=7
  Cardiotoxicity_Cardiotoxicity-10.csv                    rows=  1547  cols=7
  Cardiotoxicity_Cardiotoxicity-30.csv                    rows=  1547  cols=7
  Cardiotoxicity_Cardiotoxicity-5.csv                     rows=  1547  cols=7
  Clinical Toxicity_Clinical toxicity.csv                 rows=  1430  cols=7
  Developmental and Reproductive Toxicity_Developmental Toxicity.csv rows=   218  cols=7
  Developmen

## Step 2 — Convert single-target → multi-target

Each TOXRIC CSV alone is one endpoint. To do multi-task learning we outer-join all 30 on `Canonical SMILES`. 
Compounds tested in some endpoints but not others get `NaN` for the missing columns. 
The masked loss in `src/trainer.py` ignores `NaN` per task during training.


In [9]:
from scripts.merge_toxric import merge_toxric_csvs

merged = merge_toxric_csvs(TOXRIC_DIR)
print('shape:', merged.shape)
print('first 3 columns:', merged.columns.tolist()[:3])
print('all endpoint columns:')
for c in merged.columns[1:]:
    print(' -', c)


shape: (36015, 31)
first 3 columns: ['smiles', 'CYP450_CYP1A2', 'CYP450_CYP2C19']
all endpoint columns:
 - CYP450_CYP1A2
 - CYP450_CYP2C19
 - CYP450_CYP2C9
 - CYP450_CYP2D6
 - CYP450_CYP3A4
 - Carcinogenicity_Carcinogenicity
 - Cardiotoxicity_Cardiotoxicity-1
 - Cardiotoxicity_Cardiotoxicity-10
 - Cardiotoxicity_Cardiotoxicity-30
 - Cardiotoxicity_Cardiotoxicity-5
 - Clinical_Toxicity_Clinical_toxicity
 - Developmental_and_Reproductive_Toxicity_Developmental_Toxicity
 - Developmental_and_Reproductive_Toxicity_Reproductive_Toxicity
 - Endocrine_Disruption_NR-AR-LBD
 - Endocrine_Disruption_NR-AR
 - Endocrine_Disruption_NR-AhR
 - Endocrine_Disruption_NR-ER-LBD
 - Endocrine_Disruption_NR-ER
 - Endocrine_Disruption_NR-PPAR-gamma
 - Endocrine_Disruption_NR-aromatase
 - Endocrine_Disruption_SR-ARE
 - Endocrine_Disruption_SR-ATAD5
 - Endocrine_Disruption_SR-HSE
 - Endocrine_Disruption_SR-MMP
 - Endocrine_Disruption_SR-p53
 - Hepatotoxicity_Hepatotoxicity
 - Irritation_and_Corrosion_Eye_Corrosi

In [10]:
# How sparse is the multi-target matrix? Per-endpoint label density.
endpoint_cols = [c for c in merged.columns if c != 'smiles']
density = (merged[endpoint_cols].notna().sum() / len(merged) * 100).sort_values(ascending=False)
print('Compounds:', len(merged))
print('Endpoints:', len(endpoint_cols))
print(f'Average label density per endpoint: {density.mean():.1f}%')
print()
print('Per-endpoint coverage (% of compounds with a label):')
for name, pct in density.head(10).items():
    print(f'  {name:<55} {pct:>5.1f}%')
print('  ...')


Compounds: 36015
Endpoints: 30
Average label density per endpoint: 14.2%

Per-endpoint coverage (% of compounds with a label):
  CYP450_CYP3A4                                            31.2%
  CYP450_CYP2D6                                            29.5%
  CYP450_CYP2C19                                           29.2%
  CYP450_CYP2C9                                            27.3%
  CYP450_CYP1A2                                            26.9%
  Mutagenicity_Ames_Mutagenicity                           20.5%
  Endocrine_Disruption_NR-AR                               19.2%
  Endocrine_Disruption_SR-ATAD5                            18.7%
  Endocrine_Disruption_NR-ER-LBD                           18.4%
  Endocrine_Disruption_SR-p53                              17.9%
  ...


In [11]:
# How many endpoints does each compound have a label for?
labels_per_compound = merged[endpoint_cols].notna().sum(axis=1)
print('Labels per compound (distribution):')
print(labels_per_compound.describe())
print()
print('Histogram (bin = number of endpoints labelled):')
print(labels_per_compound.value_counts().sort_index().head(15))


Labels per compound (distribution):
count    36015.000000
mean         4.272831
std          4.195847
min          1.000000
25%          1.000000
50%          3.000000
75%          5.000000
max         26.000000
dtype: float64

Histogram (bin = number of endpoints labelled):
1     12798
2      3088
3      4533
4      5697
5      2862
6       162
7       228
8       317
9       430
10      781
11      951
12     1659
13      887
14      573
15      399
Name: count, dtype: int64


In [12]:
# Save the merged CSV in KPGT's expected location.
OUT_DIR = ROOT / 'data' / 'kpgt-cache' / 'toxric_multitask'
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_csv = OUT_DIR / 'toxric_multitask.csv'
merged.to_csv(out_csv, index=False)
print(f'wrote {out_csv} ({out_csv.stat().st_size/1e6:.2f} MB)')


wrote /Users/abose1/Avishek_Personal_Projects/mslearn-ai-studio/labfiles/toxicity-kpgt/data/kpgt-cache/toxric_multitask/toxric_multitask.csv (3.06 MB)


## Step 3 — KPGT preprocessing

KPGT's `MoleculeDataset` expects three cached files per dataset directory:
- `{dataset}_5.pkl` — DGL graph objects (built from SMILES)
- `rdkfp1-7_512.npz` — RDKit fingerprints, 512 bits, path lengths 1–7 (sparse)
- `molecular_descriptors.npz` — 200-dim normalized 2D descriptors

Plus a split index file at `splits/{name}.npy` for train/val/test partition.

Our `scripts/preprocess.py` writes the split, then subprocesses KPGT's own `preprocess_downstream_dataset.py` to produce the three cache files. 
**Heavy deps required:** `rdkit`, `dgl`, `dgllife`, `scipy`, `numpy`.


In [13]:
# Sanity check that heavy deps are installed BEFORE we try preprocessing.
missing = []
for m in ('rdkit', 'dgl', 'dgllife', 'scipy', 'numpy'):
    try:
        __import__(m)
    except ImportError:
        missing.append(m)
if missing:
    print('[skip] missing deps:', missing)
    print('Install them, then re-run this cell. The remaining cells in this section will not work without them.')
else:
    print('[ok] all preprocess deps importable')


[ok] all preprocess deps importable


In [14]:
# Generate the split file (random 80/10/10).
import numpy as np, scipy.sparse as sp
# Use the fps array size — invalid SMILES were filtered during preprocessing,
# so fps/mds/graphs have fewer rows than the raw CSV.
n_compounds = sp.load_npz(OUT_DIR / "rdkfp1-7_512.npz").shape[0]
rng = np.random.default_rng(42)
idx = rng.permutation(n_compounds)
n_test = int(n_compounds * 0.1)
n_val  = int(n_compounds * 0.1)
test_idx = idx[:n_test]
val_idx  = idx[n_test:n_test+n_val]
train_idx = idx[n_test+n_val:]
splits_dir = OUT_DIR / 'splits'
splits_dir.mkdir(exist_ok=True)
split_path = splits_dir / 'random_0.npy'
np.save(split_path, np.array([train_idx, val_idx, test_idx], dtype=object), allow_pickle=True)
print(f'splits  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}  -> {split_path}')


splits  train=28805  val=3600  test=3600  -> /Users/abose1/Avishek_Personal_Projects/mslearn-ai-studio/labfiles/toxicity-kpgt/data/kpgt-cache/toxric_multitask/splits/random_0.npy


In [15]:
# Run KPGT's preprocess as a subprocess. Takes a few minutes for ~15k compounds.
import subprocess
kpgt_preprocess = ROOT / 'external' / 'KPGT' / 'scripts' / 'preprocess_downstream_dataset.py'
if kpgt_preprocess.exists():
    cmd = [sys.executable, str(kpgt_preprocess),
           '--data_path', str((ROOT / 'data' / 'kpgt-cache').resolve()),
           '--dataset', 'toxric_multitask',
           '--n_jobs', '4']
    print('running:', ' '.join(cmd))
    print('(this can take several minutes)')
    import os
    kpgt_root = str((ROOT / "external" / "KPGT").resolve())
    env = os.environ.copy()
    env["PYTHONPATH"] = kpgt_root
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=kpgt_root, env=env)
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
else:
    print(f'[skip] {kpgt_preprocess} not found. Did you run scripts/setup.py?')


running: /Users/abose1/Avishek_Personal_Projects/mslearn-ai-studio/aboseAzureTest/bin/python /Users/abose1/Avishek_Personal_Projects/mslearn-ai-studio/labfiles/toxicity-kpgt/external/KPGT/scripts/preprocess_downstream_dataset.py --data_path /Users/abose1/Avishek_Personal_Projects/mslearn-ai-studio/labfiles/toxicity-kpgt/data/kpgt-cache --dataset toxric_multitask --n_jobs 4
(this can take several minutes)
constructing graphs
saving graphs
extracting fingerprints
saving fingerprints
extracting molecular descriptors



In [16]:
# Confirm the three cache files now exist
for fname in ('toxric_multitask_5.pkl', 'rdkfp1-7_512.npz', 'molecular_descriptors.npz'):
    p = OUT_DIR / fname
    if p.exists():
        print(f'  [ok] {fname:<35} ({p.stat().st_size/1e6:.2f} MB)')
    else:
        print(f'  [missing] {fname}')


  [ok] toxric_multitask_5.pkl              (1917.98 MB)
  [ok] rdkfp1-7_512.npz                    (16.60 MB)
  [ok] molecular_descriptors.npz           (16.37 MB)


## Step 4 — Featurize one SMILES

Inspect exactly what the model sees for a single molecule. Uses `src/featurizer.py` which produces the same (graph, fp, md) triple KPGT's preprocess builds at scale.

Demo molecule: **aspirin** (`CC(=O)OC1=CC=CC=C1C(=O)O`).


In [17]:
from toxpkg.featurizer import featurize_smiles

smiles = 'CC(=O)OC1=CC=CC=C1C(=O)O'   # aspirin
result = featurize_smiles(smiles)
if result is None:
    print('SMILES failed to featurize.')
else:
    graph, fp, md = result
    print(f'Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')
    print(f'Graph node features stored under ndata keys: {list(graph.ndata.keys())}')
    print(f'Fingerprint: shape {tuple(fp.shape)}, dtype {fp.dtype}, sum(bits)={int(fp.sum())}')
    print(f'Descriptors: shape {tuple(md.shape)}, dtype {md.dtype}, first 5 values: {md[:5].tolist()}')


Graph: 15 nodes, 185 edges
Graph node features stored under ndata keys: ['begin_end', 'edge', 'vavn']
Fingerprint: shape (512,), dtype torch.float32, sum(bits)=271
Descriptors: shape (200,), dtype torch.float32, first 5 values: [0.9769218564033508, 0.033315509557724, 0.016904747113585472, 0.011272722855210304, 0.007219666615128517]


## Step 5 — Build the fine-tuning model

Loads KPGT's `LiGhTPredictor` with the base config (`d_g_feats=768, n_mol_layers=12, n_heads=12`), loads the pretrained `base.pth` weights, then wraps it in `KPGTMultiTaskFineTuner` which:
- Sets `backbone.predictor` to a new MLP sized for our `n_tasks` (one per TOXRIC endpoint)
- Deletes the three pretraining aux heads (`md_predictor`, `fp_predictor`, `node_predictor`)
- Routes `forward()` to `LiGhT.forward_tune(g, fp, md)`, which concatenates `[fp_vn, md_vn, atom_readout]` → 3 × 768 = 2304-dim graph representation


In [18]:
import torch
from toxpkg.model import build_pretrained_predictor, KPGTMultiTaskFineTuner
from toxpkg.config import KPGT_BASE_CONFIG

n_tasks = len(endpoint_cols)
print(f'n_tasks = {n_tasks}')
print()

pretrained_path = ROOT / 'external' / 'KPGT' / 'models' / 'pretrained' / 'base' / 'base.pth'
base = build_pretrained_predictor(pretrained_path=str(pretrained_path) if pretrained_path.exists() else None)
model = KPGTMultiTaskFineTuner(base, n_tasks=n_tasks, head_hidden_dim=256, head_dropout=0.15)

n_total = sum(p.numel() for p in model.parameters())
n_head  = sum(p.numel() for n, p in model.named_parameters() if 'predictor' in n)
n_back  = n_total - n_head
print(f'Backbone params: {n_back/1e6:.1f}M (frozen-ish, lr=1e-5)')
print(f'Head     params: {n_head/1e6:.2f}M  (lr=1e-3)')
print(f'Total:           {n_total/1e6:.1f}M')
print()
print(model.backbone.predictor)


n_tasks = 30

[load_pretrained] missing keys (2): ['node_predictor.2.weight', 'node_predictor.2.bias']...
[load_pretrained] loaded /Users/abose1/Avishek_Personal_Projects/mslearn-ai-studio/labfiles/toxicity-kpgt/external/KPGT/models/pretrained/base/base.pth
Backbone params: 89.6M (frozen-ish, lr=1e-5)
Head     params: 0.60M  (lr=1e-3)
Total:           90.2M

Sequential(
  (0): Linear(in_features=2304, out_features=256, bias=True)
  (1): Dropout(p=0.15, inplace=False)
  (2): GELU(approximate='none')
  (3): Linear(in_features=256, out_features=30, bias=True)
)


## Step 6 — Training smoke-test (one batch + one optimizer step)

Builds a DataLoader using KPGT's `MoleculeDataset` + `Collator_tune`, pulls one batch, prints all the tensor shapes, runs one forward + masked-loss + backward + optimizer step. 
Lets you confirm the wiring before committing to a multi-hour training run.


In [19]:
from toxpkg.trainer import build_dataloader, masked_loss
from toxpkg.config import TrainConfig

# Infer task types from CSV column dtypes (all 30 TOXRIC endpoints are binary)
task_types = ['classification'] * len(endpoint_cols)
cfg = TrainConfig(
    data_root='data/kpgt-cache',
    dataset_name='toxric_multitask',
    split_name='random_0',
    pretrained_path=str(pretrained_path),
    n_tasks=len(endpoint_cols),
    task_names=endpoint_cols,
    task_types=task_types,
    batch_size=8,        # small for smoke test
    device='cpu',
)

train_loader = build_dataloader(cfg, split='train')
print(f'train dataset: {len(train_loader.dataset)} compounds')
print(f'batches at batch_size={cfg.batch_size}: {len(train_loader)}')


train dataset: 28805 compounds
batches at batch_size=8: 3601


In [20]:
# Pull ONE batch and inspect every tensor.
smiles_list, g, fp, md, y = next(iter(train_loader))
print('smiles in batch:', len(smiles_list))
print(f'graph (batched): {g.number_of_nodes()} nodes, {g.number_of_edges()} edges')
print(f'fp shape : {tuple(fp.shape)} dtype {fp.dtype}')
print(f'md shape : {tuple(md.shape)} dtype {md.dtype}')
print(f'y shape  : {tuple(y.shape)} dtype {y.dtype}  (NaN entries: {int(y.isnan().sum())})')


smiles in batch: 8
graph (batched): 184 nodes, 2522 edges
fp shape : (8, 512) dtype torch.float32
md shape : (8, 200) dtype torch.float32
y shape  : (8, 30) dtype torch.float32  (NaN entries: 188)


In [21]:
# Run ONE training step. Watch the loss and confirm gradients flow.
optimizer = torch.optim.AdamW(model.param_groups(cfg.backbone_lr, cfg.head_lr, cfg.weight_decay))
model.train()

logits = model(g, fp, md)
print(f'logits shape: {tuple(logits.shape)}  (expect ({cfg.batch_size}, {cfg.n_tasks}))')
loss = masked_loss(logits, y, cfg.task_types)
print(f'loss before step: {loss.item():.4f}')

loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
optimizer.step()
optimizer.zero_grad()

with torch.no_grad():
    logits2 = model(g, fp, md)
    loss2 = masked_loss(logits2, y, cfg.task_types)
print(f'loss after  step: {loss2.item():.4f}')
print('[ok] forward + masked loss + backward + step all work end-to-end.')


logits shape: (8, 30)  (expect (8, 30))
loss before step: 17.6477
loss after  step: 5.9140
[ok] forward + masked loss + backward + step all work end-to-end.


## Step 7 — Prediction on a few SMILES

Runs the SMILES → toxicity scores pipeline using whichever model state you have in memory. 
After a real training run you'd load `checkpoints/best.pt` via `predict.load_finetuned_model(...)`; here we just use the model object from Step 6.


In [22]:
from toxpkg.predict import predict_smiles, scores_per_endpoint

demo = [
    'CC(=O)OC1=CC=CC=C1C(=O)O',        # aspirin
    'CC(=O)Nc1ccc(O)cc1',                # acetaminophen / paracetamol
    'CCO',                                # ethanol
    'C(C(=O)O)N',                         # glycine
]
model.eval()
logits, valid = predict_smiles(model, demo, device='cpu')
scores = scores_per_endpoint(logits, endpoint_cols, task_types)

for s, ok, sc in zip(demo, valid, scores if valid else []):
    if not ok:
        print(f'\n=== {s} ===  INVALID')
        continue
    print(f'\n=== {s} ===')
    top = sorted(sc.items(), key=lambda kv: -kv[1])[:5]
    for name, val in top:
        print(f'  {name:<55} prob={val:.3f}')



=== CC(=O)OC1=CC=CC=C1C(=O)O ===
  Endocrine_Disruption_SR-MMP                             prob=0.732
  Respiratory_Toxicity_Respiratory_Toxicity               prob=0.694
  Endocrine_Disruption_SR-ARE                             prob=0.586
  Endocrine_Disruption_SR-ATAD5                           prob=0.582
  Cardiotoxicity_Cardiotoxicity-30                        prob=0.578

=== CC(=O)Nc1ccc(O)cc1 ===
  Irritation_and_Corrosion_Eye_Irritation                 prob=0.740
  Endocrine_Disruption_SR-MMP                             prob=0.711
  Respiratory_Toxicity_Respiratory_Toxicity               prob=0.684
  Cardiotoxicity_Cardiotoxicity-30                        prob=0.580
  Carcinogenicity_Carcinogenicity                         prob=0.560

=== CCO ===
  Irritation_and_Corrosion_Eye_Irritation                 prob=0.675
  Mutagenicity_Ames_Mutagenicity                          prob=0.666
  Cardiotoxicity_Cardiotoxicity-30                        prob=0.648
  CYP450_CYP1A2             

## Step 8 — Plain-English explanation via Claude

Sends the SMILES + per-endpoint scores to Claude via your Foundry endpoint and asks for a Markdown-formatted health summary. 
Reuses the same Anthropic + DefaultAzureCredential auth pattern as `labfiles/foundry-chat/python/chat-app/chat-app.py`.

**Requires** a `.env` with `FOUNDRY_BASE_URL` and `CLAUDE_MODEL` set, plus working Azure credentials.


In [23]:
from toxpkg.explainer import explain_predictions

# Use aspirin's scores (first row from Step 7)
if scores:
    aspirin_scores = scores[0]
    task_type_map = dict(zip(endpoint_cols, task_types))
    try:
        text = explain_predictions(
            smiles=demo[0],
            predictions=aspirin_scores,
            task_types=task_type_map,
        )
        from IPython.display import Markdown, display
        display(Markdown(text))
    except Exception as e:
        print(f'[explainer error] {type(e).__name__}: {e}')
        print('Check .env (FOUNDRY_BASE_URL, CLAUDE_MODEL) and Azure credentials.')


[explainer error] RuntimeError: FOUNDRY_BASE_URL not set. Copy .env.example to .env and fill it in.
Check .env (FOUNDRY_BASE_URL, CLAUDE_MODEL) and Azure credentials.


## Done

If everything ran without error: the merged multi-target CSV is at `data/kpgt-cache/toxric_multitask/toxric_multitask.csv`, the KPGT caches are alongside it, the model class loads, a training step works, predictions emerge from any SMILES, and Claude wraps them in plain English.

Phase D (Azure ML submission) builds on top of this — taking the same `src/trainer.py::train()` function and running it on a GPU compute cluster.
